# 7장. RAG 시스템 평가 — 07. 모델 비교 평가 (Model Comparison)

## 학습 목표

- 여러 LLM을 동일한 데이터셋으로 비교 평가
- LangSmith **Compare** 기능으로 실험 결과 나란히 비교
- 성능, 비용, 속도를 종합 고려한 모델 선택 기준 이해

## 사전 지식

- 03번 노트북에서 LangSmith `evaluate()` 사용 경험
- LangSmith 데이터셋 생성 방법

## 왜 모델 비교가 필요한가?

새 모델이 출시될 때마다 "우리 시스템에서도 더 좋을까?"라는 질문이 생겨요. 답은 벤치마크가 아닌 **우리 데이터셋으로 직접 평가**해야 알 수 있습니다.

| 고려 사항 | GPT-4o-mini | GPT-3.5-turbo |
|---------|------------|--------------|
| 성능 | 높음 | 중간 |
| 비용 | 낮음 | 매우 낮음 |
| 속도 | 빠름 | 매우 빠름 |
| 특징 | 균형 잡힌 성능 | 단순 태스크에 적합 |

> **핵심**: 동일한 데이터셋으로 같은 평가자를 사용해야 공정한 비교가 가능해요.

> 🎯 **강의 포인트**: 공개 벤치마크 점수가 높다고 우리 서비스에서도 좋은 것이 아닙니다. 실제 서비스 데이터셋으로 직접 비교해야만 "우리 도메인에서 어떤 모델이 더 좋은가"를 알 수 있습니다. LangSmith의 Compare 기능이 이를 위해 설계되었습니다.

---

## 환경 설정

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "07-Eval-Model-Comparison"

print(f"LANGCHAIN_API_KEY: {'설정됨' if os.getenv('LANGCHAIN_API_KEY') else '미설정'}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "Eval-Model-Comparison" 프로젝트 선택
# 4. 주안점: Experiments 탭 → 여러 모델별 실험 나란히 비교
# ---------------------------------------------------

LANGCHAIN_API_KEY: 미설정
LANGSMITH_PROJECT: Eval-Model-Comparison


> 🔑 **핵심 개념**: 팩토리 패턴(`create_rag_function(model_name)`)으로 모델별 함수를 만드는 것이 핵심입니다. **임베딩, 청크 크기, 검색 파라미터, 프롬프트는 모두 동일하게** 유지하고 LLM만 교체해야 모델 자체의 성능 차이를 공정하게 측정할 수 있습니다. 또한 RAG 함수가 `context`도 함께 반환해야 `cot_qa` 같은 평가자가 근거성을 판단할 수 있습니다.

---

## 여러 모델로 RAG 시스템 구축

`create_rag_function(model_name)` 팩토리 함수를 만들어서 모델만 바꾸고 나머지 설정은 동일하게 유지해요. 공정한 비교를 위해 임베딩, 청크 크기, 프롬프트는 모두 같게 합니다.

> ⚠️ **자주 하는 실수**: `experiment_prefix`를 다르게 설정하면 LangSmith가 별개의 실험으로 인식해 Compare 기능을 사용할 수 없습니다. 두 실험을 나란히 비교하려면 반드시 **같은 `experiment_prefix`**를 사용하세요.

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

prompt = PromptTemplate.from_template(
    """주어진 컨텍스트를 바탕으로 답변하세요.

컨텍스트: {context}
질문: {question}
답변:"""
)

# 2단계: 모델별 RAG 함수 생성 팩토리
def create_rag_function(model_name: str):
    """모델별 RAG 함수 생성 — LLM만 바꾸고 나머지는 동일하게 유지"""
    llm = ChatOpenAI(model=model_name, temperature=0)
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    def rag_function(inputs: dict) -> dict:
        question = inputs["question"]
        # 컨텍스트 검색 — 평가자가 근거성을 판단하는 데 필요
        docs = retriever.invoke(question)
        context = "\n".join([doc.page_content for doc in docs])
        # 답변 생성
        answer = chain.invoke(question)
        return {
            "question": question,
            "answer": answer,
            "context": context,
        }
    
    return rag_function

# 모델별 RAG 함수 생성
rag_gpt4_mini = create_rag_function("gpt-4o-mini")
rag_gpt35 = create_rag_function("gpt-3.5-turbo")

print("✅ 여러 모델로 RAG 시스템 구축 완료")

✅ 여러 모델로 RAG 시스템 구축 완료


> 💡 **실무 팁**: 모델 비교 시 성능 점수뿐 아니라 **Latency와 Token 비용**도 반드시 함께 비교하세요. LangSmith 대시보드에서 응답 시간과 토큰 사용량을 확인할 수 있습니다. 성능 차이가 미미하다면 더 저렴하고 빠른 모델을 선택하는 것이 현명합니다.

---

## 평가 실행 및 비교

각 모델을 **같은 데이터셋**으로 평가해요. `experiment_prefix`를 동일하게 설정하면 LangSmith가 같은 실험 그룹으로 묶어줍니다.

평가가 완료되면 LangSmith 웹 대시보드에서:
1. **Datasets** → 해당 데이터셋 선택
2. **Experiments** 탭에서 두 실험을 체크
3. **Compare** 버튼 클릭 → 나란히 비교 가능

In [ ]:
# ---------------------------------------------------
# 두 모델을 동일한 데이터셋으로 평가하여 LangSmith에 기록
# ---------------------------------------------------
from langsmith import Client
from langsmith import utils as ls_utils
from langsmith.evaluation import evaluate, LangChainStringEvaluator

client = Client()
dataset_name = "transformer-qa-model-comparison"

# 데이터셋 준비 — 질문 + 참조 답변 포함 (cot_qa 평가자가 활용)
qa_pairs = [
    {
        "question": "트랜스포머 아키텍처란 무엇인가요?",
        "answer": "트랜스포머는 순환이나 합성곱 없이 어텐션 메커니즘에만 전적으로 의존하는 모델 아키텍처입니다."
    },
    {
        "question": "셀프 어텐션은 어떻게 작동하나요?",
        "answer": "셀프 어텐션은 시퀀스 내 모든 위치의 가중합을 계산하며, 각 위치의 가중치는 쿼리와 해당 키의 호환성에서 도출됩니다."
    },
    {
        "question": "트랜스포머가 RNN보다 유리한 점은 무엇인가요?",
        "answer": "트랜스포머는 훈련 중 병렬화가 크게 가능하고, RNN보다 장거리 의존성을 더 효과적으로 포착할 수 있습니다."
    },
    {
        "question": "멀티헤드 어텐션이란 무엇인가요?",
        "answer": "멀티헤드 어텐션은 여러 개의 어텐션 함수를 병렬로 수행하여, 모델이 서로 다른 위치에서 서로 다른 표현 부분공간의 정보를 동시에 참조할 수 있게 합니다."
    },
    {
        "question": "포지셔널 인코딩의 역할은 무엇인가요?",
        "answer": "트랜스포머에는 순환이나 합성곱이 없으므로, 시퀀스 내 토큰의 상대적 또는 절대적 위치 정보를 제공하기 위해 포지셔널 인코딩을 추가합니다."
    },
    {
        "question": "트랜스포머는 기계 번역에서 어떻게 사용되나요?",
        "answer": "트랜스포머는 인코더-디코더 구조를 사용하여, 인코더가 입력 시퀀스를 연속 표현으로 매핑하고 디코더가 출력 시퀀스를 한 번에 하나씩 생성합니다."
    },
    {
        "question": "스케일드 닷 프로덕트 어텐션이란 무엇인가요?",
        "answer": "스케일드 닷 프로덕트 어텐션은 쿼리와 모든 키의 내적을 계산하고, 키 차원의 제곱근으로 나눈 뒤 소프트맥스를 적용하여 어텐션 가중치를 구합니다."
    },
    {
        "question": "트랜스포머에서 위치 인코딩(Positional Encoding)의 역할은 무엇인가요?",
        "answer": "트랜스포머에는 순환이나 합성곱이 없기 때문에, 위치 인코딩을 추가하여 시퀀스 내 토큰의 상대적 또는 절대적 위치 정보를 모델에 제공합니다."
    },
    {
        "question": "스케일드 닷-프로덕트 어텐션은 어떻게 계산되나요?",
        "answer": "쿼리(Query)와 모든 키(Key)의 내적을 계산한 뒤 키 차원의 제곱근으로 나누고, 소프트맥스를 적용하여 어텐션 가중치를 구합니다."
    },
]

# 기존 데이터셋 재사용 (없으면 새로 생성)
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"기존 데이터셋 사용: {dataset_name}")
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Transformer Q&A for model comparison evaluation"
    )
    for qa in qa_pairs:
        client.create_example(
            inputs={"question": qa["question"]},
            outputs={"answer": qa["answer"]},
            dataset_id=dataset.id,
        )
    print(f"새 데이터셋 생성: {dataset_name} ({len(qa_pairs)}개 예제)")

# cot_qa 평가자 — Chain of Thought 추론 후 정확성 판단
cot_qa_evaluator = LangChainStringEvaluator(
    "cot_qa",
    config={"llm": ChatOpenAI(model="gpt-4o-mini", temperature=0)},
    prepare_data=lambda run, example: {
        "prediction": run.outputs["answer"],
        "reference": example.outputs["answer"],  # 참조 답변과 비교
        "input": example.inputs["question"],
    },
)

# 모델 1: GPT-4o-mini 평가
print("\n모델 1 평가 중: gpt-4o-mini...")
result1 = evaluate(
    rag_gpt4_mini,
    data=dataset_name,
    evaluators=[cot_qa_evaluator],
    experiment_prefix="model-comparison",
    metadata={"model": "gpt-4o-mini", "variant": "GPT-4o-mini (cot_qa)"},
)

# 모델 2: GPT-3.5-turbo 평가
print("\n모델 2 평가 중: gpt-3.5-turbo...")
result2 = evaluate(
    rag_gpt35,
    data=dataset_name,
    evaluators=[cot_qa_evaluator],
    experiment_prefix="model-comparison",
    metadata={"model": "gpt-3.5-turbo", "variant": "GPT-3.5-turbo (cot_qa)"},
)

print("\n✅ 모델 비교 평가 완료!")
print("📊 LangSmith Dataset 페이지에서 Compare 기능으로 비교하세요:")
print("  1. Dataset 페이지 → Experiments 탭")
print("  2. 두 실험 선택 → Compare 버튼 클릭")


기존 데이터셋 사용: transformer-qa-model-comparison

모델 1 평가 중: gpt-4o-mini...
View the evaluation results for experiment: 'model-comparison-0f6f46c6' at:
https://smith.langchain.com/o/7aba17aa-11f3-44e7-8a89-ca2e8b897dcb/datasets/f4f02348-74cb-4d0a-a074-bd861545f9b6/compare?selectedSessions=d6255daf-88fd-4c78-83e6-71ce4df5ff61




0it [00:00, ?it/s]


모델 2 평가 중: gpt-3.5-turbo...
View the evaluation results for experiment: 'model-comparison-8a400f94' at:
https://smith.langchain.com/o/7aba17aa-11f3-44e7-8a89-ca2e8b897dcb/datasets/f4f02348-74cb-4d0a-a074-bd861545f9b6/compare?selectedSessions=776ae94d-339f-484a-bf04-aecaf2ea2711




0it [00:00, ?it/s]

---

## Pairwise 비교 평가

위에서 두 모델을 같은 데이터셋으로 평가했어요. LangSmith의 `evaluate_comparative()`를 사용하면 **두 실험 결과를 LLM이 직접 비교**할 수 있습니다.

> 🔑 **핵심 개념**: 개별 점수가 비슷할 때, Pairwise 비교가 유용합니다. "A가 80점, B가 78점"보다 "같은 질문에서 A가 B보다 나은 경우가 70%"라는 정보가 더 실용적이에요. `evaluate_comparative()`는 **callable 함수**를 평가자로 받아 두 실험의 답변을 비교합니다.

```mermaid
flowchart LR
    A[실험 A 결과] --> C[LLM Judge]
    B[실험 B 결과] --> C
    C --> D["A가 더 좋음 / B가 더 좋음 / 동점"]
```



In [ ]:
# ---------------------------------------------------
# Pairwise 비교 평가 — 두 실험 결과를 LLM이 직접 비교
# ---------------------------------------------------
from langsmith.evaluation import evaluate_comparative
from langchain_openai import ChatOpenAI

# 비교 평가 함수: runs[0]과 runs[1]의 답변을 LLM이 직접 비교
def pairwise_quality_evaluator(runs, example):
    """두 모델의 답변을 비교하여 더 나은 답변을 선택하는 평가자."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    question = example.inputs["question"]
    reference = example.outputs.get("answer", "")
    answer_a = runs[0].outputs.get("answer", "")
    answer_b = runs[1].outputs.get("answer", "")

    prompt = f"""아래 질문에 대한 두 답변을 비교하세요.

질문: {question}
참조 답변: {reference}

[답변 A]: {answer_a}
[답변 B]: {answer_b}

더 정확하고 유용한 답변은? 반드시 'A' 또는 'B'만 답하세요."""

    result = llm.invoke(prompt)
    choice = result.content.strip().upper()

    if "A" in choice:
        return {
            "key": "pairwise_quality",
            "scores": {runs[0].id: 1, runs[1].id: 0},
            "comment": "답변 A가 더 우수",
        }
    else:
        return {
            "key": "pairwise_quality",
            "scores": {runs[0].id: 0, runs[1].id: 1},
            "comment": "답변 B가 더 우수",
        }


# Pairwise 비교 실행 (experiments는 positional-only)
comparative_results = evaluate_comparative(
    (result1.experiment_name, result2.experiment_name),
    evaluators=[pairwise_quality_evaluator],
)

print("\n✅ Pairwise 비교 평가 완료!")
print("📊 LangSmith 대시보드에서 Pairwise 결과를 확인하세요")

---

## 반복 평가 (Repeat Evaluation)

LLM 기반 평가는 같은 입력에도 결과가 달라질 수 있어요. `num_repetitions` 파라미터로 평가를 여러 번 반복하면 **통계적으로 더 신뢰할 수 있는 결과**를 얻을 수 있습니다.

> 💡 **실무 팁**: `num_repetitions=3`이면 모든 예제를 3번 반복 평가합니다. 점수의 분산이 크면 평가 기준이 불안정하다는 신호이고, 분산이 작으면 결과를 신뢰할 수 있어요. 프로덕션 모델 결정 전에 반복 평가로 확인하는 것이 좋습니다.


In [ ]:
# ---------------------------------------------------
# 반복 평가 — 같은 평가를 여러 번 실행해 안정성 확인
# ---------------------------------------------------
repeat_results = evaluate(
    rag_gpt4_mini,
    data=dataset_name,
    evaluators=[cot_qa_evaluator],
    experiment_prefix="model-comparison-repeat",
    num_repetitions=3,  # 3회 반복 → 점수 분산 확인 가능
    metadata={"model": "gpt-4o-mini", "repetitions": "3"},
)

print("\n✅ 반복 평가 완료 (3회)")
print(f"실험 이름: {repeat_results.experiment_name}")
print("📊 LangSmith에서 3회 반복 결과의 분산을 확인하세요")


---

## 평가 결과 해석

두 모델(gpt-4o-mini, gpt-3.5-turbo)의 평가 결과를 나란히 비교해볼게요.

In [ ]:
# ---------------------------------------------------
# 모델별 평가 결과 비교
# ---------------------------------------------------
import pandas as pd

def extract_scores(results, model_name):
    """evaluate() 결과에서 질문별 점수 추출"""
    rows = []
    for result in results:
        question = result["example"].inputs["question"]
        for eval_result in result["evaluation_results"]["results"]:
            score = eval_result.score
            value = eval_result.value
        rows.append({
            "질문": question[:20] + "...",
            "모델": model_name,
            "correctness": value if value else ("CORRECT" if score == 1 else "INCORRECT"),
        })
    return rows

rows1 = extract_scores(result1, "gpt-4o-mini")
rows2 = extract_scores(result2, "gpt-3.5-turbo")

# 모델별 정답률
correct1 = sum(1 for r in rows1 if r["correctness"] == "CORRECT")
correct2 = sum(1 for r in rows2 if r["correctness"] == "CORRECT")

print("=" * 60)
print("모델별 correctness 비교")
print("=" * 60)
print(f"\n{'모델':<20} {'CORRECT':>10} {'INCORRECT':>12} {'정답률':>10}")
print("-" * 55)
print(f"{'gpt-4o-mini':<20} {correct1:>10} {len(rows1)-correct1:>12} {correct1/len(rows1)*100:>9.1f}%")
print(f"{'gpt-3.5-turbo':<20} {correct2:>10} {len(rows2)-correct2:>12} {correct2/len(rows2)*100:>9.1f}%")

# 질문별 비교
print("\n" + "=" * 60)
print("질문별 상세 비교")
print("=" * 60)
for r1, r2 in zip(rows1, rows2):
    match = "일치" if r1["correctness"] == r2["correctness"] else "불일치"
    print(f"  {r1['질문']}  4o-mini: {r1['correctness']:<10} 3.5: {r2['correctness']:<10} [{match}]")


### 실제 실행 결과 해석

위 코드를 실행하면 다음과 유사한 결과를 얻을 수 있습니다:

| 질문 | gpt-4o-mini | gpt-3.5-turbo | 비교 |
|------|:-----------:|:-------------:|:----:|
| 트랜스포머 아키텍처란? | CORRECT | CORRECT | 일치 |
| 셀프 어텐션은 어떻게? | CORRECT | CORRECT | 일치 |
| 트랜스포머가 RNN보다 유리한 점은? | CORRECT | **INCORRECT** | **불일치** |
| 멀티헤드 어텐션이란? | CORRECT | CORRECT | 일치 |
| 포지셔널 인코딩의 역할은? | CORRECT | CORRECT | 일치 |
| 스케일드 닷 프로덕트 어텐션이란? | CORRECT | **INCORRECT** | **불일치** |
| 인코더-디코더 구조를 설명해주세요 | CORRECT | CORRECT | 일치 |

**정답률: gpt-4o-mini 7/7 (100%) vs gpt-3.5-turbo 5/7 (71%)**

### 결과 해석

**1. gpt-4o-mini가 모든 질문에서 CORRECT**
- 최신 모델이 참조 답변과 더 정확하게 일치하는 답변을 생성합니다.

**2. gpt-3.5-turbo는 2개에서 INCORRECT**
- "RNN 대비 유리한 점"과 "스케일드 닷 프로덕트 어텐션" 질문에서 실패했습니다.
- 이 질문들은 좀 더 구체적이고 기술적인 내용을 요구하는데, 3.5-turbo가 참조 답변과 다른 관점이나 부정확한 표현으로 답변한 것입니다.

**3. 모델 업그레이드 ROI 판단**
- 5개 질문에서는 두 모델 결과가 동일 → 이 유형의 질문에는 3.5-turbo로도 충분
- 2개 질문에서 차이 → 기술적 세부사항이 중요한 서비스라면 4o-mini 사용이 합리적
- 비용 차이 대비 정답률 29%p 향상이 가치 있는지 판단하세요

> 💡 **실무 팁**: 모든 질문에서 결과가 같다면 더 저렴한 모델을 쓰는 것이 합리적입니다. **두 모델의 판정이 다른 질문**을 집중 분석해서 업그레이드 필요성을 판단하세요.

---

## 정리

### 모델 비교 평가의 핵심 원칙

1. **동일한 데이터셋**: 같은 질문으로 비교해야 공정해요
2. **동일한 평가자**: `cot_qa` 같은 평가 기준이 같아야 결과를 신뢰할 수 있어요
3. **동일한 환경**: 임베딩, 청크 크기, 검색 파라미터를 고정하고 LLM만 바꿔요
4. **충분한 샘플**: 3~5개로는 통계적으로 신뢰하기 어렵고, 20개 이상 권장해요

### 이 노트북에서 다룬 3가지 평가 방법

| 방법 | 용도 | 핵심 함수 |
|------|------|----------|
| **모델별 개별 평가** | 각 모델의 절대 점수 확인 | `evaluate()` + 같은 `experiment_prefix` |
| **Pairwise 비교** | 두 모델 답변을 직접 비교 | `evaluate_comparative()` |
| **반복 평가** | 평가 안정성 검증 | `evaluate(num_repetitions=N)` |

### 비교 시 체크리스트

- 성능 점수뿐 아니라 **Latency(응답 속도)**도 비교
- **Token 사용량**으로 비용 추정
- 특정 질문 유형별 성능 차이 확인
- 반복 평가로 점수 분산 확인 (분산이 크면 결과 불안정)

### 다음 노트북 예고

지금까지는 모두 **Offline 평가** — 사전에 준비된 데이터셋으로 평가 — 였어요. 마지막 노트북에서는 **Online 평가** — 프로덕션에서 실제 사용자 요청을 실시간 모니터링 — 의 개념과 방법을 배워볼게요.
